In [0]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline

# 1. Extração massiva de dados mantendo o LCLid
query_ml = """
SELECT 
    f.dim_household_lclid,
    t.date_time,
    t.hour,
    t.day AS day_of_week, -- (1 a 7)
    w.temp_bin_10,
    f.energy_spent AS label -- No Spark ML, o que queremos prever chama-se 'label'
FROM smart_meters.gold.fact_consumption f
INNER JOIN smart_meters.gold.dim_time t ON f.dim_time_id = t.id
INNER JOIN smart_meters.gold.dim_weather w ON f.sk_weather = w.sk_weather
"""

# Criar o DataFrame Spark (NÃO usamos .toPandas() aqui, fica tudo no Spark!)
df_spark = spark.sql(query_ml)

# Lidar com nulos por segurança
df_spark = df_spark.dropna(subset=['temp_bin_10', 'label'])

In [0]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import OneHotEncoder

indexer_temp = StringIndexer(inputCol="temp_bin_10", outputCol="temp_bin_index")

# Depois de criares os teus indexers, adicionas o Encoder:
encoder = OneHotEncoder(
    inputCols=["temp_bin_index", "hour", "day_of_week"], 
    outputCols=["temp_vec", "hour_vec", "day_vec"]
)

# O teu Assembler vai usar as novas colunas 'vec'
assembler = VectorAssembler(
    inputCols=["temp_vec", "hour_vec", "day_vec"], 
    outputCol="features"
)

lr = LinearRegression(featuresCol="features", labelCol="label")
pipeline = Pipeline(stages=[indexer_temp, encoder, assembler, lr])

In [0]:
# 1. Treinar o modelo com os teus dados
print("A treinar o modelo de Regressão Linear... (Vai ser rápido!)")
modelo_treinado = pipeline.fit(df_spark)

# 2. Gerar as previsões (O Spark cria automaticamente a coluna 'prediction')
df_predictions = modelo_treinado.transform(df_spark)

In [0]:
from pyspark.sql.functions import col

# 3. Limpar a vista e criar timestamp completo
# date_time já contém o timestamp completo hora a hora
df_final = df_predictions.select(
    col("date_time").alias("full_timestamp"),
    col("dim_household_lclid"),
    col("temp_bin_10"),
    col("label").alias("actual_energy"),
    col("prediction").alias("predicted_energy")
)

# 4. VER OS DADOS: O comando mágico do Databricks
display(df_final.limit(10))

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

# Preparar DataFrame com colunas renomeadas para o avaliador
df_eval = df_final.select(
    col("actual_energy").alias("label"),
    col("predicted_energy").alias("prediction")
)

# Criar avaliador de regressão
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction"
)

# Calcular R² (coeficiente de determinação)
r2 = evaluator.evaluate(df_eval, {evaluator.metricName: "r2"})

# Calcular RMSE (Root Mean Squared Error)
rmse = evaluator.evaluate(df_eval, {evaluator.metricName: "rmse"})

# Calcular MAE (Mean Absolute Error)
mae = evaluator.evaluate(df_eval, {evaluator.metricName: "mae"})

# Mostrar resultados
print("="*60)
print("MÉTRICAS DE PRECISÃO DO MODELO")
print("="*60)
print(f"R² (R-squared):           {r2:.4f}")
print(f"  → Quanto mais próximo de 1, melhor o ajuste")
print(f"  → {r2*100:.2f}% da variação é explicada pelo modelo\n")
print(f"RMSE (Erro Quadrático):   {rmse:.4f} kWh")
print(f"  → Erro médio das previsões\n")
print(f"MAE (Erro Absoluto):      {mae:.4f} kWh")
print(f"  → Diferença média entre real e previsto")
print("="*60)

In [0]:
%sql
DROP VIEW IF EXISTS smart_meters.gold.energy_predictions_view;

DROP TABLE IF EXISTS smart_meters.gold.energy_predictions_temp;


In [0]:
# Criar uma view Gold no Unity Catalog
# Em Spark Connect, precisamos primeiro guardar como tabela temporária, depois criar a view

table = "smart_meters.gold.energy_predictions"
df_final.write.mode("overwrite").saveAsTable(table)

print(f"✓ Podes aceder no Tableau usando: {table}")

In [0]:
%sql
CREATE OR REPLACE TABLE smart_meters.gold.energy_predictions
SELECT 
    full_timestamp,
    dim_household_lclid,
    temp_bin_10,
    actual_energy,
    predicted_energy,
    h.acorn_group
FROM smart_meters.gold.energy_predictions
INNER JOIN smart_meters.gold.dim_household h
ON smart_meters.gold.energy_predictions.dim_household_lclid = h.lclid

In [0]:
%sql
SELECT * 
FROM smart_meters.gold.energy_predictions
WHERE 
    dim_household_lclid = 'MAC000002' 
    AND full_timestamp >= '2013-01-01' 
    AND full_timestamp < '2013-01-02'